# Forest Prediction EDA

This notebook contains a compact EDA for the forest cover type classification task. It focuses on the points required by the subject: dataset structure, target balance, missing values, descriptive statistics, visual checks, correlations, and short conclusions that feed into feature engineering and model selection.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
sys.path.append(str(PROJECT_ROOT / "scripts"))

from preprocessing_feature_engineering import TARGET_COLUMN, create_engineered_features

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

In [ ]:
train_path = PROJECT_ROOT / "data" / "train.csv"
test_path = PROJECT_ROOT / "data" / "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=[TARGET_COLUMN])
eda_df = create_engineered_features(X_train).assign(Cover_Type=train_df[TARGET_COLUMN])

train_df.head()

## Shape And Basic Structure

In [ ]:
pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "rows": [len(train_df), len(test_df)],
        "columns": [train_df.shape[1], test_df.shape[1]],
    }
)

In [ ]:
train_df.info()

In [ ]:
train_df.dtypes.value_counts().rename_axis("dtype").to_frame("count")

## Target Distribution

In [ ]:
target_distribution = train_df[TARGET_COLUMN].value_counts().sort_index()
display(target_distribution.to_frame("count"))

plt.figure(figsize=(8, 4))
sns.barplot(x=target_distribution.index, y=target_distribution.values, color="#2f6f4f")
plt.title("Cover Type Distribution in train.csv")
plt.xlabel("Cover Type")
plt.ylabel("Count")
plt.tight_layout()

## Summary Statistics And Missing Values

In [ ]:
continuous_columns = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]

train_df[continuous_columns].describe().T

In [ ]:
missing_values = train_df.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

## Feature Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_columns = [
    "Elevation",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Horizontal_Distance_To_Fire_Points",
]

for ax, column in zip(axes.flat, plot_columns):
    sns.histplot(train_df[column], bins=30, ax=ax, color="#4c7b9d")
    ax.set_title(column)

plt.tight_layout()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=eda_df, x="Cover_Type", y="Elevation", color="#d9a441")
plt.title("Elevation by Cover Type")
plt.tight_layout()

## Engineered Features

In [ ]:
eda_df[["Distance_To_Hydrology", "Fire_Road_Distance_Diff"]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(eda_df["Distance_To_Hydrology"], bins=30, ax=axes[0], color="#3c8c73")
axes[0].set_title("Distance_To_Hydrology")
sns.histplot(eda_df["Fire_Road_Distance_Diff"], bins=30, ax=axes[1], color="#9a5c39")
axes[1].set_title("Fire_Road_Distance_Diff")
plt.tight_layout()

## Correlations

In [ ]:
correlation_columns = [
    "Elevation",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Horizontal_Distance_To_Fire_Points",
    "Distance_To_Hydrology",
    "Fire_Road_Distance_Diff",
]

corr = eda_df[correlation_columns].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap for Continuous and Engineered Features")
plt.tight_layout()

## Short Conclusions

- The dataset is fully labeled and already numeric, with no missing values in the training file.
- The target distribution is imbalanced enough to justify `stratify=y` in the Train(1) / Test(1) split and in cross-validation.
- Elevation clearly varies by cover type, which supports using tree-based models and linear models over the full numeric space.
- The engineered hydrology distance combines horizontal and vertical hydrology information into one physically interpretable measure.
- The fire-points minus roadways feature captures a relationship between two distance variables without exploding the feature space.
- Because KNN, SVM, and Logistic Regression are scale-sensitive, scaling is applied only to those pipelines during model selection.